In [ ]:
import os
import gc
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import h3

In [ ]:
#==========================================
# 1. Ammount of days up
# ==========================================
print("\nLoading inventory and calculating station days...")

output_file = "s3_inventory.csv"

df = pd.read_csv(output_file, usecols=['network', 'station', 'year', 'yearday', 'lat', 'lon'])

df_stations = df.groupby(['network', 'station', 'lat', 'lon']).size().reset_index(name='days')

del df
gc.collect()

print(f" -> Found {len(df_stations):,} unique stations globally.")

In [ ]:
df_stations.head()

In [ ]:
# ==========================================
# 2. Grid making and sammpling
# ==========================================
# Define H3 Resolution. 
RESOLUTION = 3
print(f"\nSnapping stations to H3 equal-area grid (Resolution {RESOLUTION})...")

try:
    df_stations['hex_id'] = df_stations.apply(lambda row: h3.latlng_to_cell(row['lat'], row['lon'], RESOLUTION), axis=1)
except AttributeError:
    df_stations['hex_id'] = df_stations.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], RESOLUTION), axis=1)

print("Filtering for the highest-uptime station per grid cell...")

df_subset = df_stations.sort_values(['hex_id', 'days'], ascending=[True, False]) \
                          .drop_duplicates(subset='hex_id', keep='first') \
                          .reset_index(drop=True)

print(f" -> Downsampled to {len(df_subset):,} stations.")

In [ ]:
# ==========================================
# 3. Settting distances (60km - 6000km)
# ==========================================
print("\nBuilding spatial index for pairing...")

EARTH_RADIUS_KM = 6371.0

MIN_DIST_KM = 60.0
MAX_DIST_KM = 6000.0

coords_rad = np.radians(df_subset[['lat', 'lon']].values)


tree = BallTree(coords_rad, metric='haversine', leaf_size=40)


In [ ]:

# ---------------------------------------------------------
# 4. Pairing
# ---------------------------------------------------------

# To radians
max_radius_rad = MAX_DIST_KM / EARTH_RADIUS_KM
min_radius_rad = MIN_DIST_KM / EARTH_RADIUS_KM

indices_max = tree.query_radius(coords_rad, r=max_radius_rad)
indices_min = tree.query_radius(coords_rad, r=min_radius_rad)

paired_data = []

for i in range(len(df_subset)):
    
    valid_neighbors = np.setdiff1d(indices_max[i], indices_min[i])
    
    unique_valid_neighbors = valid_neighbors[valid_neighbors > i]
    
    sta1 = df_subset.iloc[i]
    
    for j in unique_valid_neighbors:
        sta2 = df_subset.iloc[j]
        
       
        lat1, lon1 = coords_rad[i]
        lat2, lon2 = coords_rad[j]
        
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
        c = 2 * np.arcsin(np.sqrt(a))
        distance_km = EARTH_RADIUS_KM * c
        
        paired_data.append({
            'net1': sta1['network'],
            'sta1': sta1['station'],
            'lat1': sta1['lat'],
            'lon1': sta1['lon'],
            'days1': sta1['days'],
            'net2': sta2['network'],
            'sta2': sta2['station'],
            'lat2': sta2['lat'],
            'lon2': sta2['lon'],
            'days2': sta2['days'],
            'distance_km': round(distance_km, 2)
        })

df_pairs = pd.DataFrame(paired_data)

print(f" -> Success! Found {len(df_pairs):,} valid station pairs.")

pairs_file = "global_station_pairs10k.csv"
df_pairs.to_csv(pairs_file, index=False)

print("finished")